In [1]:
import pandas as pd
df = pd.read_parquet('data/local_generations/generations.parquet')
print('=== First 3 rows ===')
for i in range(3):
    print(f'--- Row {i} ---')
    for col in df.columns:
        val = str(df[col].iloc[i])[:300]
        print(f'  {col}: {repr(val)}')

=== First 3 rows ===
--- Row 0 ---
  generated_id: 'gen-000000'
  prompt: 'Once upon a time, there was a little girl who liked to taste things. One day, she found a pin and picked it up. She wanted to taste it, so she licked'
  response: '<think>\n\n</think>\n\nz_4: little girl\nz_3: little girl liked tasting\nz_2: Once upon a time there was a girl\nz_1: Once upon a time there was a girl who liked to taste things and licked a pin'
--- Row 1 ---
  generated_id: 'gen-000001'
  prompt: 'the park. One day, Tim took his machine to the park to sail it in the pond. As Tim was sailing his machine, a girl named Sue came up to him. She'
  response: '<think>\n\n</think>\n\nz_4: Tim Sue\nz_3: Tim sailed in pond\nz_2: Tim sailed in pond and Sue approached him\nz_1: She smiled and asked if Tim wanted to try a ride together on the park path'
--- Row 2 ---
  generated_id: 'gen-000002'
  prompt: 'and content. The other kids saw how helpful Anna was and thanked her for her help. The moral of the story i

In [2]:
from src.config import load_config
from pathlib import Path

config = load_config("config/latent_generation.yaml")
df = pd.read_parquet("data/local_generations/generations.parquet")
prompts = df["prompt"].tolist()
outputs = df["response"].tolist()
print(f"Loaded {len(outputs)} outputs")

Loaded 100000 outputs


In [3]:
from src.generation.dataset import verify_and_filter_outputs, flatten_for_c2f

/home/ax46/project/coarse-to-fine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
prompts_filtered, outputs_filtered, stats = verify_and_filter_outputs(prompts, outputs, config)
print(stats)

Verification Statistics:
  Total Processed: 100000
  Passed: 61984
  Failed: 38016
  Pass Rate: 61.98%
  Failure Breakdown:
    - z_1: expected 16 words, got 17: 15493
    - z_1: expected 16 words, got 15: 15278
    - z_2: expected 8 words, got 9: 2887
    - z_2: expected 8 words, got 7: 2875
    - z_1: expected 16 words, got 18: 477
    - z_3: expected 4 words, got 5: 447
    - z_1: expected 16 words, got 14: 318
    - z_3: expected 4 words, got 3: 124
    - z_4: expected 2 words, got 3: 91
    - z_4: expected 2 words, got 1: 13
    - z_1: expected 16 words, got 19: 4
    - z_1: expected 16 words, got 13: 4
    - z_2: expected 8 words, got 6: 2
    - Failed to parse layer structure: 2
    - z_2: expected 8 words, got 10: 1


In [5]:
if outputs_filtered:
    c2f_path = flatten_for_c2f(prompts_filtered, outputs_filtered, Path("data/local_generations"))
    print(f"C2F data saved to: {c2f_path}")
else:
    print("No outputs passed verification")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  6.80ba/s]

C2F data saved to: data/local_generations/c2f_train.parquet


In [6]:
import pandas as pd
df = pd.read_parquet('data/local_generations/c2f_train.parquet')
print('=== First 3 rows ===')
for i in range(3):
    print(f'--- Row {i} ---')
    for col in df.columns:
        val = str(df[col].iloc[i])[:300]
        print(f'  {col}: {repr(val)}')

=== First 3 rows ===
--- Row 0 ---
  text: 'Tim Sue Tim sailed in pond Tim sailed in pond and Sue approached him She smiled and asked if Tim wanted to try a ride together on the park path the park. One day, Tim took his machine to the park to sail it in the pond. As Tim was sailing his machine, a girl named Sue came up to him. She'
--- Row 1 ---
  text: 'kind helpful kind helpful brings happiness The other kids saw how helpful Anna was The moral of the story is that being kind and helpful to others will bring you and content. The other kids saw how helpful Anna was and thanked her for her help. The moral of the story is that being kind and helpful t'
--- Row 2 ---
  text: 'help animals help wild animals people She told people what tiger said and surprised The people decided to help the animals find their homes and protect them from danger together help the wild animals. She went to the people in the town and told them what the tiger said. The people were surprised too'
